# transformerRafa

Este notebook usa BERT como o transformer principal e mantém o pré-processamento e a estrutura do pipeline:

1. limpa o texto
2. substitui URLs, leis/decretos, número de processo e telefone por tokens especiais
3. divide os dados em treino e teste
4. treina Word2Vec no texto de treino
5. prepara os rótulos e tokeniza os textos para o BERT
6. executa grid search com validação cruzada estratificada de 5 folds
7. seleciona a configuração pela acurácia média dos folds
8. retreina com todo o conjunto de treino e reporta a acurácia no teste

No BERT, embeddings e positional encoding são tratados internamente pelo modelo. A validação cruzada é aplicada ao conjunto de treino; o conjunto de teste fica reservado para a avaliação final.

In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold

from gensim.models import Word2Vec
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

In [ ]:
# === 1. Leitura do dataset ===
df = pd.read_excel('train.xlsx')
print('Shape:', df.shape)
print(df.head())


In [ ]:
# === 2. Limpeza e substituições ===
def clean_text(text):
    text = str(text).lower()

    # URLs -> token especial
    text = re.sub(r'https?://\S+|www\.\S+', ' LINK_URL ', text)

    # Lei / Decreto -> token especial
    text = re.sub(
        r'\b(?:lei|decreto)\s*n[ºo]?\s*\d{1,3}(?:[\.,]\d{1,3})*(?:/\d{4})?\b',
        ' LEI_DECRETO ',
        text,
        flags=re.IGNORECASE,
    )

    # Processo -> token especial
    text = re.sub(r'\b\d{5}\.\d{3,6}/\d{4}-\d{2}\b', ' NUMERO_PROCESSO ', text)

    # Telefone -> token especial
    text = re.sub(
        r'\b(?:\+?\d{2}\s?)?(?:\(?\d{2}\)?\s?\d{4,5}\s?\d{4}|\d{4}\s?\d{4})\b',
        ' NUMERO_TELEFONE ',
        text,
    )

    # Remove pontuação, preservando tokens especiais
    text = re.sub(r'[^a-z0-9_\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['resp_text'] = df['resp_text'].fillna('').apply(clean_text)
df['clarity'] = df['clarity'].astype(str)
print(df[['resp_text', 'clarity']].head())
print('Classes:', sorted(df['clarity'].unique()))


In [ ]:
# === 3. Divisão treino/teste ===
X = df['resp_text'].tolist()
y = df['clarity'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Treino:', len(X_train))
print('Teste:', len(X_test))


In [ ]:
# === 4. Treinando Word2Vec (baseline opcional / estrutura do pipeline) ===
sentences = [text.split() for text in X_train if text.strip()]

w2v = Word2Vec(
    sentences=sentences,
    vector_size=128,
    window=5,
    min_count=2,
    sg=1,
    workers=1,
    seed=42
)

print('Vocab size:', len(w2v.wv.key_to_index))
print('Exemplo de embedding:', w2v.wv['link_url'] if 'link_url' in w2v.wv else 'token não encontrado')


In [ ]:
# === 5. Preparação para BERT ===
labels = sorted(set(y))
label_to_id = {label: idx for idx, label in enumerate(labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

train_df = pd.DataFrame({'text': X_train, 'label': [label_to_id[v] for v in y_train]})
test_df = pd.DataFrame({'text': X_test, 'label': [label_to_id[v] for v in y_test]})

print('Labels:', labels)
print('Label map:', label_to_id)


In [ ]:
# === 6. Tokenização do BERT ===
model_name = 'neuralmind/bert-base-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding=True, max_length=128)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column('label', 'labels')
test_ds = test_ds.rename_column('label', 'labels')

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(train_ds[0])


In [ ]:
# === 7. Modelo BERT para classificação ===
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id_to_label,
    label2id=label_to_id
)

print(model)


In [ ]:
# === 8. Grid Search com validação cruzada ===
def compute_accuracy(eval_pred):
    logits, true_labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': float(np.mean(predictions == true_labels))}

n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
grid = [
    {'lr': 2e-5, 'batch_size': 8},
    {'lr': 2e-5, 'batch_size': 16},
    {'lr': 3e-5, 'batch_size': 8},
    {'lr': 3e-5, 'batch_size': 16},
]

results = []

for cfg in grid:
    fold_accuracies = []
    print(f"\n=== Testando cfg: {cfg} ===")

    for fold, (train_idx, val_idx) in enumerate(cv.split(np.zeros(len(y_train)), y_train), start=1):
        print(f'Fold {fold}/{n_splits}')
        fold_train_ds = train_ds.select(train_idx.tolist())
        fold_val_ds = train_ds.select(val_idx.tolist())
        fold_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(labels),
            id2label=id_to_label,
            label2id=label_to_id,
        )
        training_args = TrainingArguments(
            output_dir=f"./tmp_bert_{cfg['lr']}_{cfg['batch_size']}_fold_{fold}",
            eval_strategy='epoch',
            save_strategy='no',
            learning_rate=cfg['lr'],
            per_device_train_batch_size=cfg['batch_size'],
            per_device_eval_batch_size=cfg['batch_size'],
            num_train_epochs=1,
            weight_decay=0.01,
            logging_steps=100,
            disable_tqdm=False,
        )

        trainer = Trainer(
            model=fold_model,
            args=training_args,
            train_dataset=fold_train_ds,
            eval_dataset=fold_val_ds,
            compute_metrics=compute_accuracy,
        )
        trainer.train()
        metrics = trainer.evaluate()
        fold_accuracy = metrics['eval_accuracy']
        fold_accuracies.append(fold_accuracy)
        print(f'Acurácia do fold {fold}: {fold_accuracy:.4f}')

    mean_accuracy = float(np.mean(fold_accuracies))
    std_accuracy = float(np.std(fold_accuracies))
    results.append({
        **cfg,
        'mean_accuracy': mean_accuracy,
        'std_accuracy': std_accuracy,
    })
    print(f'Acurácia média: {mean_accuracy:.4f} (+/- {std_accuracy:.4f})')

print('\n=== RESULTADO DA VALIDAÇÃO CRUZADA ===')
for result in sorted(results, key=lambda item: item['mean_accuracy'], reverse=True):
    print(result)

best_cfg = max(results, key=lambda item: item['mean_accuracy'])
print('\nMelhor configuração:', best_cfg)

# Retreina com todo o conjunto de treino e avalia uma única vez no conjunto de teste.
final_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id_to_label,
    label2id=label_to_id,
)
final_args = TrainingArguments(
    output_dir='./bert_final_best',
    eval_strategy='no',
    save_strategy='no',
    learning_rate=best_cfg['lr'],
    per_device_train_batch_size=best_cfg['batch_size'],
    per_device_eval_batch_size=best_cfg['batch_size'],
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    disable_tqdm=False,
)
final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=train_ds,
    compute_metrics=compute_accuracy,
)
final_trainer.train()
test_metrics = final_trainer.evaluate(test_ds)
print('Acurácia final no teste:', test_metrics['eval_accuracy'])

## Interpretação final

- O pré-processamento específico do problema foi mantido.
- O grid search testa combinações de learning rate e tamanho do batch.
- Cada configuração é comparada por validação cruzada estratificada de 5 folds, usando acurácia.
- A melhor configuração é retreinada em todo o conjunto de treino e avaliada uma vez no conjunto de teste.
- O transformer usado é o BERT (`neuralmind/bert-base-portuguese-cased`).
- O Word2Vec permanece no pipeline como referência, mas a classificação final é feita pelo BERT.